# Análisis de apariencia externa — imagen individual

*Versión de Traitly utilizada en este tutorial: 0.1.0*

En este tutorial, demostraremos cómo realizar el análisis de apariencia externa de frutos utilizando `FruitExternalAnalyzer`, una herramienta para extraer medidas de morfología y color a partir de una sola imagen.

Primero, cargamos la clase `FruitExternalAnalyzer` desde Traitly y la imagen a analizar.

In [ ]:
from traitly.fruit_phenotyping import FruitExternalAnalyzer


path_img = '~/ext_analysis_sample1.jpg'

pic_test = FruitExternalAnalyzer(path_img)

pic_test.load_image()

Después ejecutamos `setup_measurements()` para detectar las referencias de tamaño en la imagen (círculos negros).

In [ ]:
pic_test.setup_measurements()

Ahora generemos las máscaras de frutos con `generate_fruit_mask()` usando el valor de color de fondo predeterminado `white` y veamos qué objetos en la imagen fueron detectados como frutos con `detect_fruits()`.

Como podemos ver en los gráficos generados a continuación, la mayoría de los frutos han sido segmentados efectivamente. Sin embargo, notamos que algunos frutos no fueron detectados (no tienen el contorno verde) por `detect_fruits()`. En estos casos, necesitamos modificar algunos parámetros para mejorar tanto las máscaras como la detección; abordemos eso a continuación.

In [ ]:
pic_test.generate_fruit_mask(background_color='white')

pic_test.detect_fruits(
    plot=True, plot_size=(5,5),
    contour_thickness=7,
)    

En `generate_fruit_mask()`, las máscaras de los frutos a veces pueden mostrar hendiduras o regiones de las orillas sin segmentar correctamente. Para cerrar estos espacios, podemos usar el parámetro `apply_convex_hull=True`, que aplica un [convex hull](https://www.geeksforgeeks.org/dsa/convex-hull-algorithm/) alrededor del contorno del fruto, garantizando un resultado más suave y cerrado. Además, el parámetro `kernel_blur=5` también ayuda a definir mejor los contornos de los frutos, difuminando y simplificando los colores de la imágen, lo que facilita la segmentación. Tener una buena definición de los contornos es fundamental, ya que los cálculos se basan en ellos, y cualquier hueco en el contorno impacta directamente en los análisis posteriores. Opcionalmente, se puede aplicar `erosion_px=3` para eliminar algunos píxeles de alrededor del contorno cuyo color podría verse afectado por el reflejo del fondo. La erosión también ayuda a eliminar porciones del fondo que podrían estar incluidas en la máscara y que sesgarían las estimaciones de color del fruto.

En `detect_fruits()`, `min_fruit_circularity=0.4` garantiza que capturemos todos los frutos al reducir el umbral de circularidad, ya que algunos tienen una forma más alargada.

In [ ]:
pic_test.generate_fruit_mask(background_color='white',
                             apply_convex_hull=True,
                             kernel_blur=5,
                             erosion_px=1)

pic_test.detect_fruits(
    plot=True, plot_size=(5,5),
    contour_thickness=7,
    min_fruit_circularity=0.4
)

Ahora que los frutos han sido correctamente segmentados y detectados, podemos realizar los análisis morfológicos y de color.

In [ ]:
pic_test.analyze_morphology(display_table=False,
                            plot=True,
                            plot_size=(5,5))

pic_test.analyze_color(display_table=False,
                       plot=False)

Al ejecutar `analyze_morphology()` y `analyze_color()`, se generará el objeto `resultados` que contiene el método `save_all()`, el cual podemos llamar de la siguiente manera para guardar tanto los archivos CSV con los resultados de cada análisis como la imagen anotada. En la imagen anotada, podemos verificar que tanto los frutos como las referencias de tamaño han sido correctamente detectadas.

In [ ]:
pic_test.results.save_all()

Opcionalmente, puedes guardar los parámetros e información de la sesión de tu análisis con `save_parameters()` para garantizar la reproducibilidad de tus análisis futuros o para utilizarlos en el procesamiento por lotes.

In [ ]:
pic_test.save_parameters()